### Feature Engineering

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
da = pd.read_csv('../data/interim/dailyActivity_merged.csv')
da['ActivityDate'] = pd.to_datetime(da['ActivityDate'], format='%m/%d/%Y')

In [3]:
da['is_worn'] = ~((da['SedentaryMinutes'] >= 1440) | (da['Calories'] == 0))
da['weekday'] = da['ActivityDate'].dt.day_name()
da['is_weekend'] = da['ActivityDate'].dt.dayofweek >= 5
da['ActiveMinutes'] = da['VeryActiveMinutes'] + da['FairlyActiveMinutes'] + da['LightlyActiveMinutes']
da['ActiveRatio'] = da['ActiveMinutes'] / 1440
da['SedentaryRatio'] = da['SedentaryMinutes'] / 1440
da['VeryActiveRatio'] = da['VeryActiveMinutes'] / da['ActiveMinutes'].replace(0, np.nan)

In [4]:
sleep = pd.read_csv('../data/interim/sleepDay_merged.csv')
sleep['SleepDay'] = pd.to_datetime(sleep['SleepDay']).dt.normalize()
sleep = sleep.drop_duplicates(subset=['Id','SleepDay'])
sleep = sleep.rename(columns={'SleepDay':'ActivityDate'})

In [5]:
da = da.merge(sleep[['Id','ActivityDate','TotalMinutesAsleep','TotalTimeInBed']],
              on=['Id','ActivityDate'], how='left')

print('병합 전 dailyActivity 행수 기준 검증 필요:', len(da), '행 (1+2구간 합산)')
da.to_csv('../data/processed/daily_features.csv', index=False)
da.head()

병합 전 dailyActivity 행수 기준 검증 필요: 1397 행 (1+2구간 합산)


,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,...,ExportPeriod,is_worn,weekday,is_weekend,ActiveMinutes,ActiveRatio,SedentaryRatio,VeryActiveRatio,TotalMinutesAsleep,TotalTimeInBed
0,1503960366,2016-03-25,11004,7.11,7.11,0.0,2.57,0.46,4.07,0.0,...,1구간,True,Friday,False,250,0.173611,0.558333,0.132000,NaN,NaN
1,1503960366,2016-03-26,17609,11.55,11.55,0.0,6.92,0.73,3.91,0.0,...,1구간,True,Saturday,True,380,0.263889,0.408333,0.234211,NaN,NaN
2,1503960366,2016-03-27,12736,8.53,8.53,0.0,4.66,0.16,3.71,0.0,...,1구간,True,Sunday,True,329,0.228472,0.420139,0.170213,NaN,NaN
3,1503960366,2016-03-28,13231,8.93,8.93,0.0,3.19,0.79,4.95,0.0,...,1구간,True,Monday,False,283,0.196528,0.750000,0.137809,NaN,NaN
4,1503960366,2016-03-29,12041,7.85,7.85,0.0,2.16,1.09,4.61,0.0,...,1구간,True,Tuesday,False,299,0.207639,0.529861,0.093645,NaN,NaN


In [6]:
daily = pd.read_csv('../data/processed/daily_features.csv', parse_dates=['ActivityDate'])
worn = daily[daily['is_worn']]

total_days = daily.groupby('Id')['ActivityDate'].nunique()
worn_days = worn.groupby('Id')['ActivityDate'].nunique()

user_summary = pd.DataFrame({'ObservedDays': total_days, 'WornDays': worn_days}).fillna(0)
user_summary['WearRate'] = user_summary['WornDays'] / user_summary['ObservedDays']

agg = worn.groupby('Id').agg(
    AvgSteps=('TotalSteps','mean'),
    AvgCalories=('Calories','mean'),
    AvgActiveMinutes=('ActiveMinutes','mean'),
    AvgSedentaryRatio=('SedentaryRatio','mean'),
    AvgVeryActiveMinutes=('VeryActiveMinutes','mean'),
    AvgSleepMinutes=('TotalMinutesAsleep','mean'),
    SleepRecordedDays=('TotalMinutesAsleep', lambda s: s.notna().sum()),
)

weekday_avg = worn[~worn['is_weekend']].groupby('Id')['TotalSteps'].mean().rename('AvgSteps_Weekday')
weekend_avg = worn[worn['is_weekend']].groupby('Id')['TotalSteps'].mean().rename('AvgSteps_Weekend')

user_summary = user_summary.join(agg).join(weekday_avg).join(weekend_avg)

weight = pd.read_csv('../data/interim/weightLogInfo_merged.csv')
weight_freq = weight.groupby('Id').size().rename('WeightLogCount')
user_summary = user_summary.join(weight_freq).fillna({'WeightLogCount': 0})

hr_ids = set(pd.read_csv('../data/interim/heartrate_seconds_merged.csv', usecols=['Id'])['Id'])
user_summary['HasHeartRate'] = user_summary.index.isin(hr_ids)

user_summary = user_summary.round(2)
user_summary.to_csv('../data/processed/user_summary.csv')
print('사용자 수:', len(user_summary))
user_summary.head()

사용자 수: 35


,ObservedDays,WornDays,WearRate,AvgSteps,AvgCalories,AvgActiveMinutes,AvgSedentaryRatio,AvgVeryActiveMinutes,AvgSleepMinutes,SleepRecordedDays,AvgSteps_Weekday,AvgSteps_Weekend,WeightLogCount,HasHeartRate
Id,,,,,,,,,,,,,,
1503960366,49,48,0.98,12179.37,1845.65,284.29,0.57,38.39,359.0,26,12245.14,12014.93,3.0,False
1624580081,49,49,1.00,5167.20,1433.78,150.64,0.88,5.66,NaN,0,4706.36,6352.21,0.0,False
1644430081,40,40,1.00,7780.92,2837.58,228.55,0.78,10.88,294.0,4,6587.96,10564.50,0.0,False
1844505072,42,32,0.76,3747.55,1657.52,167.97,0.75,0.39,652.0,3,3576.25,4204.33,0.0,False
1927972279,42,29,0.69,1819.00,2252.77,87.37,0.78,1.37,472.5,6,1446.95,2842.12,2.0,False


In [7]:
hs = pd.read_csv('../data/interim/hourlySteps_merged.csv')
hs['ActivityHour'] = pd.to_datetime(hs['ActivityHour'])

hc = pd.read_csv('../data/interim/hourlyCalories_merged.csv')
hc['ActivityHour'] = pd.to_datetime(hc['ActivityHour'])

hi = pd.read_csv('../data/interim/hourlyIntensities_merged.csv')
hi['ActivityHour'] = pd.to_datetime(hi['ActivityHour'])

hourly = hs.merge(hc[['Id','ActivityHour','Calories']], on=['Id','ActivityHour'], how='left') \
           .merge(hi[['Id','ActivityHour','TotalIntensity','AverageIntensity']], on=['Id','ActivityHour'], how='left')

hourly.to_csv('../data/processed/hourly_pattern.csv', index=False)
print('저장 완료:', len(hourly), '행')
hourly.head()

저장 완료: 47233 행


,Id,ActivityHour,StepTotal,ExportPeriod,Calories,TotalIntensity,AverageIntensity
0,1503960366,2016-03-12 00:00:00,0,1구간,48,0,0.0
1,1503960366,2016-03-12 01:00:00,0,1구간,48,0,0.0
2,1503960366,2016-03-12 02:00:00,0,1구간,48,0,0.0
3,1503960366,2016-03-12 03:00:00,0,1구간,48,0,0.0
4,1503960366,2016-03-12 04:00:00,0,1구간,48,0,0.0
